# **Suggestion Algorithm**
The goal of this section is to implement a simple algorithm that, based on the genre chosen by the user and based on his taste, finds the user with most similar taste and suggests movies based on what the most similar user has liked.

In [1]:
import pandas as pd

In [ ]:
ratings = pd.read_csv('/Users/beatricecitterio/ratings.csv')
movies = pd.read_csv('/Users/beatricecitterio/movies.csv')
# change paths with where you stored the data

In [3]:
ratings = ratings.drop(columns='timestamp')

In [4]:
movie_counts = ratings['movieId'].value_counts().reset_index()
movie_counts.columns = ['movieId', 'count']

In [6]:
movie_counts # create this df so that we know the popularity of each movie (i.e. how many times it has been rated)

,movieId,count
0,318,102929
1,356,100296
2,296,98409
3,2571,93808
4,593,90330
...,...,...
84427,288825,1
84428,288467,1
84429,287221,1
84430,284087,1


In [7]:
genres = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'Any Genre']

In [8]:
top_movies = {}
for genre in genres:
    if genre == 'Any Genre':
        mov = movies.merge(movie_counts, on = 'movieId').sort_values(by= 'count', ascending = False)
        top_movies[genre] = mov.nlargest(20, 'count')
    else: 
        genre_movies = movies[movies['genres'].str.contains(genre, case=False)]
        genre_movies = genre_movies.merge(movie_counts, on = 'movieId').sort_values(by= 'count', ascending = False)
        top_movies[genre] = genre_movies.nlargest(20, 'count')


This dictionary stores the 20 most popular (i.e. most rated) movies for each genre.

In [9]:
top_movies['Any Genre']

,movieId,title,genres,count
314,318,"Shawshank Redemption, The (1994)",Crime|Drama,102929
351,356,Forrest Gump (1994),Comedy|Drama|Romance|War,100296
292,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,98409
2480,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808
585,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,90330
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,85010
2867,2959,Fight Club (1999),Action|Crime|Drama|Thriller,77332
475,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233
522,527,Schindler's List (1993),Drama|War,73849
4888,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,73122


In [10]:
top_movies['Thriller']

,movieId,title,genres,count
52,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,98409
431,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808
110,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,90330
487,2959,Fight Club (1999),Action|Crime|Drama|Thriller,77332
89,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233
9,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,67750
8,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,63298
111,608,Fargo (1996),Comedy|Crime|Drama|Thriller,58031
2456,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,57931
134,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,57224


Now, the user is asked to fill out a form in order to understand his taste. First, he needs to specify the genre he is interested in (among those proposed by us). Suppose this is given, we will call it 'genre' and it will be a string. He will also be asked how long he wants the survey to be (5, 10 or 20 movies to rate). Given these, the algorithm will output the list of movies he needs to rate.

In [ ]:
def movies_to_rate(genre: str, n: int):
    return list(top_movies[genre][:n].title)

Given this list of movies, the user will give a rating from 0 to 5 to each one of them. If he hasn't seen the movie, he must put 'Not seen'. Suppose this rating is given as an array, called new_rating, of size n.

In [21]:
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np

In [22]:
def suggestion(new_rating: list, genre: str, n: int):
    new_rating_dict = {}
    for i in range(n):
        new_rating_dict[list(top_movies[genre][:n].movieId)[i]] = new_rating[i]

    new_rating_filtered =  {k: v for k, v in new_rating_dict.items() if v != 'Not seen'}
    filtered_ratings = ratings[ratings['movieId'].isin(new_rating_filtered.keys())]

    pivot_df = filtered_ratings.pivot(index='userId', columns='movieId', values='rating').fillna(2.5)

    new_user_ratings = pd.DataFrame([new_rating_filtered], index=['new_user'])
    dissimilarities = euclidean_distances(pivot_df, new_user_ratings)

    most_similar_user = pivot_df.index[np.argmin(dissimilarities)]

    print(f'Most similar user ID: {most_similar_user}')

    user_ratings = ratings[ratings['userId'] == most_similar_user]
    movies_by_genre = movies[movies['genres'].str.contains(genre, case=False)]
    user_ratings = user_ratings[user_ratings['movieId'].isin(movies_by_genre.movieId)]

    user_ratings_filtered = user_ratings.sort_values(by = 'rating', ascending=False)

    suggested_movies = user_ratings_filtered[~user_ratings_filtered['movieId'].isin(new_rating_filtered.keys())]
    suggested_movies = suggested_movies.merge(movie_counts).sort_values(by = ['rating', 'count'], ascending=False)
    final_suggestions = suggested_movies[:3]
    final_suggestions = movies[movies['movieId'].isin(final_suggestions.movieId)].title
    return final_suggestions


## **EXAMPLE**

In [15]:
top10 = top_movies['Thriller'][:10]
top10_titles = list(top_movies['Thriller'][:10].title)

My input is the following:

In [12]:
new_rating = [5, 'Not seen', 4.5, 5, 'Not seen', 'Not seen', 5, 'Not seen', 4.5, 'Not seen']

In [13]:
new_rating_dict = {}
for i in range(10):
    new_rating_dict[list(top10.movieId)[i]] = new_rating[i]

In [14]:
new_rating_filtered =  {k: v for k, v in new_rating_dict.items() if v != 'Not seen'}


In [15]:
new_rating_filtered

{296: 5, 593: 4.5, 2959: 5, 47: 5, 79132: 4.5}

In [16]:
filtered_ratings = ratings[ratings['movieId'].isin(new_rating_filtered.keys())]

In [17]:
new_rating_filtered.values()

dict_values([5, 4.5, 5, 5, 4.5])

In [18]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

In [19]:
pivot_df = filtered_ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
pivot_df

movieId,47,296,593,2959,79132
userId,,,,,
1,0.0,0.0,3.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0
5,3.0,1.0,3.0,0.0,0.0
7,0.0,5.0,0.0,0.0,0.0
...,...,...,...,...,...
200944,4.5,4.5,4.0,5.0,5.0
200945,4.0,0.5,4.0,5.0,5.0
200946,0.0,5.0,5.0,0.0,0.0


In [20]:
new_user_ratings = pd.DataFrame([new_rating_filtered], index=['new_user'])

In [21]:
dissimilarities = euclidean_distances(pivot_df, new_user_ratings)

In [22]:
most_similar_user = pivot_df.index[np.argmin(dissimilarities)]

print(f'Most similar user ID: {most_similar_user}')

Most similar user ID: 8


In [23]:
user_8 = pivot_df.loc[8]
user_8

movieId
47       5.0
296      4.5
593      5.0
2959     5.0
79132    4.5
Name: 8, dtype: float64

In [24]:
user_ratings = ratings[ratings['userId'] == 8]
thriller_movies = movies[movies['genres'].str.contains('Thriller', case=False)]
user_ratings = user_ratings[user_ratings['movieId'].isin(thriller_movies.movieId)]

In [25]:
user_ratings

,userId,movieId,rating
470,8,32,4.0
471,8,47,5.0
473,8,296,4.5
475,8,593,5.0
476,8,608,5.0
482,8,2712,4.0
484,8,2959,5.0
486,8,4011,3.5
487,8,4226,4.5
489,8,6016,4.5


In [26]:
user_ratings_filtered = user_ratings.sort_values(by = 'rating', ascending=False)
# QUA NON METTERE NECESSARIAMENTE RATING DI 5 BASTA METTERE RATING IN ORDINE DI GRANDEZZA

In [27]:
user_ratings_filtered

,userId,movieId,rating
471,8,47,5.0
475,8,593,5.0
476,8,608,5.0
484,8,2959,5.0
492,8,27773,5.0
473,8,296,4.5
487,8,4226,4.5
489,8,6016,4.5
495,8,44555,4.5
498,8,79132,4.5


In [28]:
suggested_movies = user_ratings_filtered

In [29]:
suggested_movies

,userId,movieId,rating
471,8,47,5.0
475,8,593,5.0
476,8,608,5.0
484,8,2959,5.0
492,8,27773,5.0
473,8,296,4.5
487,8,4226,4.5
489,8,6016,4.5
495,8,44555,4.5
498,8,79132,4.5


In [30]:
suggested_movies = suggested_movies[~suggested_movies['movieId'].isin(new_rating_filtered.keys())]

In [31]:
suggested_movies = suggested_movies.merge(movie_counts).sort_values(by = ['rating', 'count'], ascending=False)

In [32]:
final_suggestions = suggested_movies[:3]
final_suggestions = movies[movies['movieId'].isin(final_suggestions.movieId)].title

In [33]:
final_suggestions

600       Fargo (1996)
4123    Memento (2000)
9338    Old Boy (2003)
Name: title, dtype: object

## **EXAMPLE 2** 

In [35]:
top10 = top_movies['Sci-Fi'][:10]

In [36]:
top10

,movieId,title,genres,count
172,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808
13,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,85010
26,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233
57,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,72151
32,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,68383
61,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,67496
68,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,61879
968,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,57931
45,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,57224
2,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,55275


In [37]:
new_rating = [4.5, 4, 3, 3.5, 'Not seen', 3.5, 3.5, 4.5, 2, 3.5]

In [38]:
new_rating_dict = {}
for i in range(10):
    new_rating_dict[list(top10.movieId)[i]] = new_rating[i]

In [39]:
new_rating_filtered =  {k: v for k, v in new_rating_dict.items() if v != 'Not seen'}


In [40]:
new_rating_filtered

{2571: 4.5,
 260: 4,
 480: 3,
 1196: 3.5,
 1210: 3.5,
 1270: 3.5,
 79132: 4.5,
 780: 2,
 32: 3.5}

In [41]:
filtered_ratings = ratings[ratings['movieId'].isin(new_rating_filtered.keys())]

In [42]:
new_rating_filtered.values()

dict_values([4.5, 4, 3, 3.5, 3.5, 3.5, 4.5, 2, 3.5])

In [43]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

In [44]:
pivot_df = filtered_ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
pivot_df

movieId,32,260,480,780,1196,1210,1270,2571,79132
userId,,,,,,,,,
1,5.0,5.0,0.0,0.0,5.0,2.0,5.0,0.0,0.0
3,0.0,4.0,4.0,4.0,4.0,4.0,4.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0
5,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
8,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,4.5
...,...,...,...,...,...,...,...,...,...
200944,0.0,4.0,0.0,0.0,3.5,4.0,4.5,5.0,5.0
200945,5.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,5.0
200946,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0


In [45]:
new_user_ratings = pd.DataFrame([new_rating_filtered], index=['new_user'])

In [46]:
dissimilarities = euclidean_distances(pivot_df, new_user_ratings)

In [47]:
most_similar_user = pivot_df.index[np.argmin(dissimilarities)]

print(f'Most similar user ID: {most_similar_user}')

Most similar user ID: 41417


In [48]:
user_8 = pivot_df.loc[41417]
user_8

movieId
32       4.0
260      4.0
480      3.0
780      3.5
1196     4.0
1210     4.0
1270     4.0
2571     2.5
79132    4.0
Name: 41417, dtype: float64

In [49]:
user_ratings = ratings[ratings['userId'] == 41417]
thriller_movies = movies[movies['genres'].str.contains('Thriller', case=False)]
user_ratings = user_ratings[user_ratings['movieId'].isin(thriller_movies.movieId)]

In [50]:
user_ratings

,userId,movieId,rating
6634703,41417,6,3.5
6634704,41417,10,2.0
6634706,41417,20,2.0
6634708,41417,32,4.0
6634710,41417,47,4.0
...,...,...,...
6635409,41417,129937,2.5
6635410,41417,130634,3.0
6635412,41417,132796,2.0
6635426,41417,143385,3.0


In [51]:
user_ratings_filtered = user_ratings.sort_values(by = 'rating', ascending=False)
# QUA NON METTERE NECESSARIAMENTE RATING DI 5 BASTA METTERE RATING IN ORDINE DI GRANDEZZA

In [52]:
user_ratings_filtered

,userId,movieId,rating
6635256,41417,73017,5.0
6635161,41417,48780,4.5
6635405,41417,122920,4.5
6635259,41417,74458,4.5
6635100,41417,8950,4.5
...,...,...,...
6635183,41417,54648,1.0
6635058,41417,6564,1.0
6635361,41417,105653,1.0
6635169,41417,51077,1.0


In [53]:
suggested_movies = user_ratings_filtered

In [54]:
suggested_movies

,userId,movieId,rating
6635256,41417,73017,5.0
6635161,41417,48780,4.5
6635405,41417,122920,4.5
6635259,41417,74458,4.5
6635100,41417,8950,4.5
...,...,...,...
6635183,41417,54648,1.0
6635058,41417,6564,1.0
6635361,41417,105653,1.0
6635169,41417,51077,1.0


In [55]:
suggested_movies = suggested_movies[~suggested_movies['movieId'].isin(new_rating_filtered.keys())]

In [56]:
suggested_movies = suggested_movies.merge(movie_counts).sort_values(by = ['rating', 'count'], ascending=False)

In [57]:
final_suggestions = suggested_movies[:3]
final_suggestions = movies[movies['movieId'].isin(final_suggestions.movieId)].title

In [58]:
final_suggestions

11165      Prestige, The (2006)
14108    Sherlock Holmes (2009)
14338     Shutter Island (2010)
Name: title, dtype: object

## **EXAMPLE 3**

In [99]:
top10 = top_movies['Action'][:10]

In [100]:
top10

,movieId,title,genres,count
345,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808
39,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,85010
400,2959,Fight Club (1999),Action|Crime|Drama|Thriller,77332
78,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233
151,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,72151
18,110,Braveheart (1995),Action|Drama|War,69482
97,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,68383
158,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,67496
940,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,67449
153,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,67408


In [109]:
new_rating = [4, 4, 5, 3, 5, 2, 3, 5, 4, 3]

In [110]:
new_rating_dict = {}
for i in range(10):
    new_rating_dict[list(top10.movieId)[i]] = new_rating[i]

In [111]:
new_rating_filtered =  {k: v for k, v in new_rating_dict.items() if v != 'Not seen'}


In [112]:
new_rating_filtered

{2571: 4,
 260: 4,
 2959: 5,
 480: 3,
 1196: 5,
 110: 2,
 589: 3,
 1210: 5,
 7153: 4,
 1198: 3}

In [113]:
filtered_ratings = ratings[ratings['movieId'].isin(new_rating_filtered.keys())]

In [114]:
new_rating_filtered.values()

dict_values([4, 4, 5, 3, 5, 2, 3, 5, 4, 3])

In [115]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

In [116]:
pivot_df = filtered_ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
pivot_df

movieId,110,260,480,589,1196,1198,1210,2571,2959,7153
userId,,,,,,,,,,
1,3.0,5.0,0.0,0.0,5.0,0.0,2.0,0.0,0.0,0.0
3,5.0,4.0,4.0,3.0,4.0,4.5,4.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0
5,4.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
...,...,...,...,...,...,...,...,...,...,...
200944,4.0,4.0,0.0,4.0,3.5,0.0,4.0,5.0,5.0,5.0
200945,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,5.0,0.5
200946,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0


In [117]:
new_user_ratings = pd.DataFrame([new_rating_filtered], index=['new_user'])

In [118]:
dissimilarities = euclidean_distances(pivot_df, new_user_ratings)

In [119]:
most_similar_user = pivot_df.index[np.argmin(dissimilarities)]

print(f'Most similar user ID: {most_similar_user}')

Most similar user ID: 182656


In [120]:
user_8 = pivot_df.loc[182656]
user_8

movieId
110     4.0
260     4.5
480     4.5
589     4.0
1196    5.0
1198    2.5
1210    3.0
2571    5.0
2959    4.5
7153    2.5
Name: 182656, dtype: float64

In [121]:
user_ratings = ratings[ratings['userId'] == 182656]
thriller_movies = movies[movies['genres'].str.contains('Action', case=False)]
user_ratings = user_ratings[user_ratings['movieId'].isin(thriller_movies.movieId)]

In [122]:
user_ratings

,userId,movieId,rating
29145132,182656,6,2.5
29145133,182656,10,3.5
29145143,182656,110,4.0
29145145,182656,145,3.0
29145149,182656,260,4.5
...,...,...,...
29146614,182656,132046,2.5
29146618,182656,134170,4.0
29146624,182656,135536,3.0
29146638,182656,138036,4.0


In [123]:
user_ratings_filtered = user_ratings.sort_values(by = 'rating', ascending=False)
# QUA NON METTERE NECESSARIAMENTE RATING DI 5 BASTA METTERE RATING IN ORDINE DI GRANDEZZA

In [124]:
user_ratings_filtered

,userId,movieId,rating
29145789,182656,6539,5.0
29145934,182656,8972,5.0
29146010,182656,33794,5.0
29145269,182656,1196,5.0
29146082,182656,49272,5.0
...,...,...,...
29146364,182656,92938,1.0
29146077,182656,48774,1.0
29146015,182656,37386,0.5
29145285,182656,1215,0.5


In [125]:
suggested_movies = user_ratings_filtered

In [126]:
suggested_movies

,userId,movieId,rating
29145789,182656,6539,5.0
29145934,182656,8972,5.0
29146010,182656,33794,5.0
29145269,182656,1196,5.0
29146082,182656,49272,5.0
...,...,...,...
29146364,182656,92938,1.0
29146077,182656,48774,1.0
29146015,182656,37386,0.5
29145285,182656,1215,0.5


In [127]:
suggested_movies = suggested_movies[~suggested_movies['movieId'].isin(new_rating_filtered.keys())]

In [128]:
suggested_movies = suggested_movies.merge(movie_counts).sort_values(by = ['rating', 'count'], ascending=False)

In [129]:
final_suggestions = suggested_movies[:3]
final_suggestions = movies[movies['movieId'].isin(final_suggestions.movieId)].title

In [130]:
final_suggestions

3480                                      Gladiator (2000)
6417     Pirates of the Caribbean: The Curse of the Bla...
10004                                 Batman Begins (2005)
Name: title, dtype: object